# HydroClaude 快速入门教程
# Quick Start Tutorial

**作者 / Author**: HydroClaude Development Team  
**日期 / Date**: 2025-10-30  
**难度 / Level**: 初级 / Beginner  
**时长 / Duration**: 15分钟

---

## 📚 教程目标 / Tutorial Objectives

在这个教程中，你将学会：
1. 创建简单的管网拓扑
2. 配置节点和管道
3. 使用Hardy Cross求解器
4. 分析计算结果
5. 可视化结果

---

## 🎯 问题描述 / Problem Description

我们将创建一个简单的供水系统：
- 1个水库（恒定水头）
- 3个用水点（Junction节点）
- 4根管道（形成树状结构）

```
R1 (水库) → P1 → J1 → P2 → J2
                 ↓
                 P3
                 ↓
                 J3
```

---

## Step 1: 导入必要的模块
## Import Required Modules

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

# 添加项目路径
sys.path.insert(0, os.path.dirname(os.getcwd()))

from network.pressure_pipe import create_pressure_pipe
from network.network_node import Junction, Reservoir
from network.network_topology import NetworkTopology
from solvers.hardy_cross_solver import HardyCrossSolver

print("✅ 模块导入成功！")
print("✅ Modules imported successfully!")

## Step 2: 创建网络拓扑
## Create Network Topology

In [ ]:
# 创建拓扑对象
topology = NetworkTopology("快速入门示例 / Quick Start Example")

print(f"✅ 创建拓扑: {topology.name}")

## Step 3: 添加节点
## Add Nodes

### 3.1 添加水库（水源）

In [ ]:
# 创建水库：标高50m，水头80m（即水面标高80m）
reservoir = Reservoir(
    node_id='R1',
    elevation=50.0,  # 地面标高 (m)
    head=80.0        # 水面标高 (m)
)

topology.add_node(reservoir)

print(f"✅ 添加水库 R1:")
print(f"   - 地面标高: {reservoir.elevation}m")
print(f"   - 水面标高: {reservoir.head}m")
print(f"   - 可用水头: {reservoir.head - reservoir.elevation}m")

### 3.2 添加用水节点（Junction）

In [ ]:
# 创建3个用水点
junctions = [
    ('J1', 45.0, 10.0),  # (节点ID, 标高m, 需水量L/s)
    ('J2', 40.0, 15.0),
    ('J3', 42.0, 12.0),
]

for jid, elev, demand_ls in junctions:
    # 需水量转换：L/s → m³/s
    demand_m3s = demand_ls / 1000.0
    
    junction = Junction(
        node_id=jid,
        elevation=elev,
        demand=demand_m3s
    )
    
    topology.add_node(junction)
    
    print(f"✅ 添加节点 {jid}: 标高={elev}m, 需水量={demand_ls}L/s")

## Step 4: 添加管道
## Add Pipes

In [ ]:
# 管道配置：(ID, 起点, 终点, 管径mm, 长度m, 局部损失系数)
pipe_configs = [
    ('P1', 'R1', 'J1', 200, 500, 1.0),  # 水库→J1
    ('P2', 'J1', 'J2', 150, 400, 0.5),  # J1→J2
    ('P3', 'J1', 'J3', 150, 350, 0.5),  # J1→J3
]

for pid, from_node, to_node, D_mm, L, K in pipe_configs:
    # 创建管道对象
    pipe = create_pressure_pipe(
        pipe_id=pid,
        diameter=D_mm / 1000.0,  # mm → m
        length=L,
        material='steel',        # 钢管
        K_minor=K               # 局部损失系数
    )
    
    # 添加到拓扑，指定起点和终点
    topology.add_pipe(pipe, from_node, to_node)
    
    print(f"✅ 添加管道 {pid}: {from_node}→{to_node}, DN{D_mm}, L={L}m")

print(f"\n📊 网络统计:")
print(f"   - 节点数量: {len(topology.nodes)}")
print(f"   - 管道数量: {len(topology.pipes)}")

## Step 5: 求解网络
## Solve Network

使用Hardy Cross方法求解管网的流量和水头分布。

In [ ]:
# 创建Hardy Cross求解器
solver = HardyCrossSolver(
    topology=topology,
    max_iter=100,      # 最大迭代次数
    tol=1e-6,          # 收敛容差
    verbose=True       # 显示详细信息
)

print("🔧 开始求解...\n")

# 求解
flows, heads = solver.solve()

# 检查收敛性
if solver.converged:
    print(f"\n✅ 求解成功！")
    print(f"   - 迭代次数: {solver.iteration}")
    print(f"   - 最大残差: {solver.residual:.2e}")
else:
    print(f"\n❌ 求解未收敛")
    print(f"   - 迭代次数: {solver.iteration}")
    print(f"   - 最大残差: {solver.residual:.2e}")

## Step 6: 分析结果
## Analyze Results

### 6.1 查看管道流量

In [ ]:
print("\n📊 管道流量分布:")
print("=" * 60)
print(f"{'管道ID':<8} {'流量(L/s)':<12} {'流速(m/s)':<12} {'方向'}")
print("=" * 60)

for pid, pipe in topology.pipes.items():
    Q = flows[pid]  # m³/s
    Q_ls = Q * 1000  # L/s
    
    # 计算流速
    A = np.pi * (pipe.D / 2) ** 2  # 断面积
    V = abs(Q) / A  # 流速
    
    # 判断流向
    direction = "→" if Q > 0 else "←"
    
    print(f"{pid:<8} {abs(Q_ls):<12.2f} {V:<12.3f} {direction}")

print("=" * 60)

### 6.2 查看节点水头和压力

In [ ]:
print("\n📊 节点水头与压力:")
print("=" * 80)
print(f"{'节点ID':<8} {'标高(m)':<12} {'水头(m)':<12} {'压力(m)':<12} {'压力(kPa)'}")
print("=" * 80)

for nid, node in topology.nodes.items():
    head = heads[nid]
    pressure_m = head - node.elevation
    pressure_kpa = pressure_m * 9.81  # 转换为kPa
    
    print(f"{nid:<8} {node.elevation:<12.2f} {head:<12.2f} "
          f"{pressure_m:<12.2f} {pressure_kpa:<12.2f}")

print("=" * 80)

# 检查压力是否满足要求
min_pressure = min(heads[nid] - node.elevation 
                   for nid, node in topology.nodes.items() 
                   if isinstance(node, Junction))

print(f"\n✅ 最小压力: {min_pressure:.2f}m")

if min_pressure >= 15:
    print("   ✓ 满足最小压力要求（≥15m）")
else:
    print("   ⚠️ 压力不足，建议增大管径或提高水源水头")

## Step 7: 可视化结果
## Visualize Results

In [ ]:
# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 子图1：管道流量
pipe_ids = list(flows.keys())
flow_values = [abs(flows[pid]) * 1000 for pid in pipe_ids]  # L/s

ax1.bar(pipe_ids, flow_values, color='steelblue', alpha=0.7, edgecolor='black')
ax1.set_ylabel('Flow Rate (L/s)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Pipe ID', fontsize=12, fontweight='bold')
ax1.set_title('Pipe Flow Distribution', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')

# 添加数值标签
for i, (pid, val) in enumerate(zip(pipe_ids, flow_values)):
    ax1.text(i, val + 0.5, f'{val:.1f}', ha='center', va='bottom', fontweight='bold')

# 子图2：节点压力
junction_ids = [nid for nid, node in topology.nodes.items() if isinstance(node, Junction)]
pressures = [heads[nid] - topology.nodes[nid].elevation for nid in junction_ids]

colors = ['green' if p >= 15 else 'orange' for p in pressures]
ax2.bar(junction_ids, pressures, color=colors, alpha=0.7, edgecolor='black')
ax2.axhline(y=15, color='red', linestyle='--', linewidth=2, label='Min Required (15m)')
ax2.set_ylabel('Pressure (m)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Node ID', fontsize=12, fontweight='bold')
ax2.set_title('Node Pressure Distribution', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')
ax2.legend()

# 添加数值标签
for i, (nid, p) in enumerate(zip(junction_ids, pressures)):
    ax2.text(i, p + 0.5, f'{p:.1f}m', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✅ 可视化完成！")

## 🎓 知识要点总结 / Key Takeaways

### 1. 管网组成
- **节点 (Nodes)**: Reservoir（水源）、Junction（用水点）、Tank（水箱）
- **管道 (Pipes)**: 连接节点，传输流体

### 2. 参数单位
- 长度：米 (m)
- 流量：m³/s（内部）或 L/s（显示）
- 压力：米水柱 (m) 或 kPa
- 管径：米 (m)（内部）或 mm（显示）

### 3. 求解方法
- **Hardy Cross法**: 适用于环状管网，迭代求解流量分配
- **收敛判据**: 残差 < 容差（如1e-6）

### 4. 工程标准
- 最小压力：≥15m（城市供水）
- 最大流速：≤3.0m/s（防止水锤）
- 最小流速：≥0.3m/s（防止沉积）

---

## 🚀 下一步学习 / Next Steps

继续学习以下教程：
1. **环状管网分析** - 学习如何处理复杂的环状拓扑
2. **多工况分析** - 分析高峰、平均、低谷等不同工况
3. **水锤分析** - 瞬态水力学分析
4. **管网优化** - 管径选型和经济性分析

---

## 📚 参考资料 / References

- HydroClaude 官方文档
- 案例库：`examples/` 目录
- API文档：`docs/` 目录

---

**祝学习愉快！/ Happy Learning!** 🎉